In [1]:
from sklearn.base import BaseEstimator
import numpy as np

class MyDummyClassifier(BaseEstimator):
    def fit(self, X, y=None):
        pass
    def predict(self, X):
        pred = np.zeros((X.shape[0], 1))
        for i in range(X.shape[0]):
            if X['Sex'].iloc[i] == 1:
                pred[i] = 0
            else:
                pred[i] = 1
        return pred

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import LabelEncoder

# 1. 빈칸(Null) 처리
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True) # 나이는 평균으로
    df['Cabin'].fillna('N', inplace=True)            # 객실은 'N'으로
    df['Embarked'].fillna('N', inplace=True)         # 항구도 'N'으로
    df['Fare'].fillna(0, inplace=True)
    return df

# 2. 불필요한 속성 제거
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 3. 문자열을 숫자로 변환 (레이블 인코딩)
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1] # 객실 번호의 첫 글자만 추출
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 4. 위 3개 함수를 순서대로 한 번에 실행하는 최종 함수
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

ti = pd.read_csv('titanic.csv')
test_ti = ti['Survived']
train_ti = ti.drop('Survived', axis=1)
train_ti = transform_features(train_ti)
train_input, test_input, train_target, test_target = train_test_split(train_ti, test_ti, test_size=0.2, random_state=42)

C:\Users\1724q\AppData\Local\Temp\ipykernel_8996\963524425.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].mean(), inplace=True) # 나이는 평균으로
C:\Users\1724q\AppData\Local\Temp\ipykernel_8996\963524425.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


In [ ]:
myclf = MyDummyClassifier()
myclf.fit(train_input, train_target)

mypred = myclf.predict(test_input)
print('정확도는: {0:.4f}'.format(accuracy_score(test_target, mypred)))
# 정답인 test_target이 앞에 나오고, 그 다음 답안인 mypred가 뒤에 나와야 올바른 accuracy score이 된다.

정확도는: 0.7821


In [5]:
from sklearn.datasets import load_digits
import numpy as np

class myfakeclf(BaseEstimator):
    def fit(self, X, y):
        pass

    # 입력값으로 들어오는 X 데이터의 세트 크기 만큼 모두 0값으로 만들어서 반환
    def predict(self, X):
        return np.zeros((len(X), 1), dtype=bool)
    
# MNIST 데이터 로딩
digits = load_digits()

# digits 번호가 7이라면 True고, 이를 astype(int)로 1로 변환, 7번이 아니라면 False고 0으로 변환
y = (digits.target == 7).astype(int)
train_input, test_input, train_target, test_target = train_test_split(digits.data, y, random_state=42)

In [7]:
print('레이블 테스트 세트 크기:', test_target.shape)
print('테스트 세트 레이블 0과 1의 분포도')
print(pd.Series(test_target).value_counts())

fakeclf = myfakeclf()
fakeclf.fit(train_input, train_target)
fakepred = fakeclf.predict(test_input)
print('모든 예측을 0으로 해도 정확도는 {0:.4f}'.format(accuracy_score(test_target, fakepred)))

레이블 테스트 세트 크기: (450,)
테스트 세트 레이블 0과 1의 분포도
0    409
1     41
Name: count, dtype: int64
모든 예측을 0으로 해도 정확도는 0.9089


In [8]:
from sklearn.metrics import confusion_matrix

confusion_matrix(test_target, fakepred)

array([[409,   0],
       [ 41,   0]])

In [9]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

def get_clf_eval(test_target, pred):
    confusion = confusion_matrix(test_target, pred)
    accuracy = accuracy_score(test_target, pred)
    precision = precision_score(test_target, pred)
    recall = recall_score(test_target, pred)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}'.format(accuracy, precision, recall))
    

In [10]:
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(train_input, train_target)
pred = lr_clf.predict(test_input)
get_clf_eval(test_target, pred)

오차 행렬
[[408   1]
 [  1  40]]
정확도: 0.9956, 정밀도: 0.9756, 재현율: 0.9756


In [12]:
pred_proba = lr_clf.predict_proba(test_input)
pred = lr_clf.predict(test_input)
print(pred_proba.shape)
print(pred_proba[:3])
pred_proba_result = np.concatenate([pred_proba, pred.reshape(-1, 1)], axis=1)
print(pred_proba_result[:3])

(450, 2)
[[9.99999999e-01 8.79149874e-10]
 [9.99999998e-01 2.45548422e-09]
 [9.99999532e-01 4.67902009e-07]]
[[9.99999999e-01 8.79149874e-10 0.00000000e+00]
 [9.99999998e-01 2.45548422e-09 0.00000000e+00]
 [9.99999532e-01 4.67902009e-07 0.00000000e+00]]


In [ ]:
# 나머지 코드는 굳이 타이핑하지 않고, 그 개념과 설계 의의를 이해하는 데 주력함

# 정확도 → 불균형 레이블을 가진 데이터에서는 사용되어서는 안되는 지표
# 오차행렬 → TF / TN / FP / FN 4개의 칸

# confusion_matrix 자체가 이미 칸의 위치가 규정된 함수이기 때문에, 이대로 외우는 게 좋겠다.

# 실제/예측                              Negative (0)                  Positive (1)

# Negative (0)

# Positive (1)

# 정밀도 (Precision) = TP / (TP + FP) ⇒ 정밀도는 우리가 양성이라고 예측한 것 중에 실제 양성의 비율. FP를 낮추는 데 초점
# 재현율 (민감도, Recall) = TP / (TP + FN) ⇒ 재현율은 실제 양성인 것 중에 우리가 양성이라고 예측한 비율. FN을 낮추는 데 초점

# 정밀도가 중요한 지표가 되는 경우: 실제 음성 데이터를 양성으로 판단하면 위험한 경우 (가령 스팸 메일 분류) precision_score()

# 재현율이 중요한 지표가 되는 경우: 실제 양성 데이터를 음성으로 판단하면 위험한 경우 (가령 암 환자 판단) recall_score()

# 이진 분류에서의 임계값(Threshold)로 Positive와 Negative를 결정한다.

# 개별 데이터별로 예측 확률을 반환하는 메서드인 predict_proba() 를 제공한다. 

# F1 스코어 = 2 * (precision * recall) / (precision + recall)
# ROC(수신자 판단 곡선) = False Positive Rate를 X축에, True Positive Rate를 Y축에 두고, FPR의 변화에 따른 TPR의 변화를 곡선 형태로 나타낸다. 
# ROC 곡선은  모델이 결과값을 예측할 때 적용하는 임계값을 조정하면 어떤 변화가 일어나는지 파악하기 위해 설계되었다. 
# 천천히 생각해보자. 일단 TPR 즉, recall 부터 생각해보면 recall은 FN을 낮춰야 좋다. 그러기 위해서는 Negative라고 판단하지 않는 게 유리하다. 
# 그래서 임계값을 낮출 유인이 생긴다. 임계값을 낮춰야 Positive라고 말할 확률이 높아지고 그러면 Negative라고 판단할 확률이 낮아진다. 
# 그런데 마냥 그 확률을 낮추는 게 좋을 수만은 없다. 그럼 실제로는 Negative인 것도 Positive라고 말하게 될 수 있기 때문이다. 이걸 조준하는 게 바로 False Positive다. 
# 그래서 이 상충관계를 시각화하여 ROC 곡선에서 타협점을 찾는 것이다.
# AUC